# 43 — Robustness Preserves the Connected Admissible Region

**Engineering question**

How can a connected admissible region remain useful under realistic perturbations?

Notebook 37 established that computation exists inside a **largest connected admissible region**.  
Notebook 43 asks how engineering preserves that region under drift, loss, noise, timing error, and calibration change.

Engineering statement:

> Robust engineering preserves not a single operating point, but the connected admissible region surrounding it.


## Setup

This notebook generates figures and data into:

```text
figures/
results/csv/
results/json/
results/43_outputs.zip
```

The final cell starts a download in Google Colab or displays a clickable link in Jupyter.


In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Region versus point

A single operating point can look good and still be fragile.

A connected admissible region is stronger because small perturbations can remain inside the working region.

Let

\[
\mathcal R
\]

represent the connected admissible region.

Let

\[
\delta(\mathcal R)
\]

represent the smallest perturbation that moves the architecture outside the admitted region.

Notebook 43 treats this margin as an engineering object.


In [ ]:
x = np.linspace(0, 1, 260)
y = np.linspace(0, 1, 260)
X, Y = np.meshgrid(x, y)

# Illustrative admissibility landscape.
squeezing_gate = 1 / (1 + np.exp(-14 * (X - 0.30)))
loss_decay = np.exp(-2.8 * Y)
overdrive = np.exp(-9.0 * np.maximum(X - 0.84, 0) ** 2)
timing_stability = np.exp(-4.5 * np.maximum(X + 0.70 * Y - 1.06, 0) ** 2)
region_shape = np.exp(-((X - 0.60) ** 2 / 0.26 + (Y - 0.22) ** 2 / 0.18))
Z = squeezing_gate * loss_decay * overdrive * timing_stability * (0.55 + 0.45 * region_shape)
Z = Z / Z.max()

fig, ax = plt.subplots(figsize=(9.5, 5.8))
im = ax.imshow(Z, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)

levels = [0.35, 0.55, 0.72]
cs = ax.contour(X, Y, Z, levels=levels, colors="black", linewidths=[1.4, 2.0, 2.8])
ax.clabel(cs, inline=True, fontsize=8)

target = (0.60, 0.22)
ax.scatter([target[0]], [target[1]], s=240, zorder=5)
ax.annotate("operating point", xy=target, xytext=(0.72, 0.34),
            arrowprops=dict(arrowstyle="->", linewidth=1.9), fontsize=10)

circle = Circle(target, 0.115, fill=False, linewidth=2.2, linestyle="--")
ax.add_patch(circle)
ax.text(0.47, 0.49, "connected\nadmissible region", ha="center", fontsize=12)
ax.text(0.68, 0.18, "robust interior", fontsize=10)
ax.text(0.18, 0.82, "under-driven", ha="center", fontsize=10)
ax.text(0.84, 0.82, "unstable /\nover-driven", ha="center", fontsize=10)
ax.text(0.86, 0.09, "loss-limited", ha="center", fontsize=10)

ax.set_title("Robustness Margin Around the Connected Admissible Region", fontsize=16)
ax.set_xlabel("squeezing / nonlinear interaction strength")
ax.set_ylabel("optical loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("admissibility score")

savefig(fig, "43_01_robustness_margin.png")
plt.show()


## 2. Drift directions

Perturbations are directional.

The same operating point may be robust to one kind of drift and fragile to another.

Typical drift directions include:

- loss increase
- phase drift
- detector noise
- timing error
- calibration shift
- pump-power fluctuation


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect("equal")
ax.axis("off")

center = np.array([0.52, 0.50])
safe_radius = 0.29
failure_radius = 0.44

ax.add_patch(Circle(center, failure_radius, fill=False, linewidth=2.0, alpha=0.35))
ax.add_patch(Circle(center, safe_radius, fill=True, alpha=0.10))
ax.add_patch(Circle(center, safe_radius, fill=False, linewidth=2.8))
ax.scatter([center[0]], [center[1]], s=280, zorder=5)

small_vectors = {
    "loss drift": np.array([-0.10, 0.14]),
    "phase drift": np.array([0.15, 0.10]),
    "timing error": np.array([-0.13, -0.12]),
    "detector noise": np.array([0.18, -0.10]),
}
large_vectors = {
    "calibration jump": np.array([0.36, 0.24]),
    "pump surge": np.array([0.40, -0.05]),
}

for label, v in small_vectors.items():
    ax.arrow(center[0], center[1], v[0], v[1],
             width=0.005, head_width=0.025, length_includes_head=True)
    ax.text(center[0] + 1.18*v[0], center[1] + 1.18*v[1], label, fontsize=9)

for label, v in large_vectors.items():
    ax.arrow(center[0], center[1], v[0], v[1],
             width=0.005, head_width=0.025, length_includes_head=True, alpha=0.55)
    ax.text(center[0] + 1.02*v[0], center[1] + 1.02*v[1], label, fontsize=9)

ax.text(center[0], center[1] - 0.06, "operating\npoint", ha="center", va="top", fontsize=9)
ax.text(center[0], center[1] + safe_radius + 0.035, "safe perturbation margin", ha="center", fontsize=10)
ax.text(center[0], center[1] - failure_radius - 0.07, "large perturbations cross a failure boundary", ha="center", fontsize=10)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Robustness Depends on Distance to Every Failure Boundary", fontsize=15)

savefig(fig, "43_02_drift_directions.png")
plt.show()


## 3. Connected region survival

A region can shrink, fragment, or disappear.

Robustness is not only area. It also involves connectivity.

The most useful regime is the one that remains connected after perturbations.


In [ ]:
def admissibility_surface(loss_shift=0.0, squeeze_shift=0.0, noise=0.0):
    xs = np.linspace(0, 1, 140)
    ys = np.linspace(0, 1, 140)
    Xs, Ys = np.meshgrid(xs, ys)
    base = np.exp(-((Xs - (0.56 + squeeze_shift)) ** 2 / 0.10 + (Ys - (0.25 + loss_shift)) ** 2 / 0.07))
    penalty = np.exp(-2.2 * Ys) * (1 / (1 + np.exp(-12 * (Xs - 0.22))))
    island = 0.42 * np.exp(-((Xs - 0.82) ** 2 / 0.015 + (Ys - 0.18) ** 2 / 0.018))
    Zs = base * penalty + island
    if noise:
        Zs = Zs + noise * (np.sin(28*Xs) * np.sin(20*Ys))
    Zs = np.clip(Zs, 0, None)
    return xs, ys, Xs, Ys, Zs / Zs.max()

cases = [
    ("baseline", 0.00, 0.00, 0.00),
    ("drifted", 0.08, -0.05, 0.00),
    ("fragmented", 0.12, -0.08, 0.10),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)
metrics = []

for ax, (name, ls, ss, nz) in zip(axes, cases):
    xs, ys, Xs, Ys, Zs = admissibility_surface(ls, ss, nz)
    ax.imshow(Zs, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)
    ax.contour(Xs, Ys, Zs, levels=[0.50], colors="black", linewidths=2.2)
    ax.set_title(name)
    ax.set_xlabel("drive")
    if ax is axes[0]:
        ax.set_ylabel("loss")
    admitted_fraction = float((Zs >= 0.50).mean())
    metrics.append({"case": name, "admitted_fraction": admitted_fraction})

fig.suptitle("Connected Region Survival Under Perturbation", fontsize=16, y=1.04)
savefig(fig, "43_03_region_survival.png")
plt.show()

region_metrics = pd.DataFrame(metrics)
region_metrics.to_csv(CSV / "43_region_survival.csv", index=False)
region_metrics.to_json(JS / "43_region_survival.json", orient="records", indent=2)
region_metrics


## 4. Architecture comparison

Different architectures preserve different margins.

The comparison here is illustrative. It summarizes the engineering idea:

```text
region size + margin + connectivity → robustness
```


In [ ]:
architectures = ["single\nresonator", "coupled\nresonators", "programmable\nmesh", "hybrid\nchip"]

comparison = pd.DataFrame({
    "architecture": ["single resonator", "coupled resonators", "programmable mesh", "hybrid chip"],
    "region_size": [0.32, 0.53, 0.78, 0.92],
    "margin": [0.25, 0.44, 0.67, 0.84],
    "connectivity": [0.38, 0.55, 0.80, 0.90],
})
comparison["robustness_score"] = comparison[["region_size", "margin", "connectivity"]].mean(axis=1)

x = np.arange(len(architectures))
width = 0.22

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.bar(x - width, comparison["region_size"], width, label="region size")
ax.bar(x, comparison["margin"], width, label="margin")
ax.bar(x + width, comparison["connectivity"], width, label="connectivity")

ax.set_xticks(x)
ax.set_xticklabels(architectures)
ax.set_ylim(0, 1.05)
ax.set_ylabel("relative score")
ax.set_title("Architecture Comparison: Robustness Is More Than One Metric", fontsize=15)
ax.legend()
ax.grid(axis="y", alpha=0.25)

comparison.to_csv(CSV / "43_architecture_comparison.csv", index=False)
comparison.to_json(JS / "43_architecture_comparison.json", orient="records", indent=2)

savefig(fig, "43_04_architecture_comparison.png")
plt.show()
comparison


## 5. Robustness landscape

Good engineering widens a plateau.

A narrow high-performing ridge can be fragile.  
A broad plateau can preserve computation under realistic drift.


In [ ]:
x = np.linspace(-1.2, 1.2, 280)
y = np.linspace(-1.2, 1.2, 280)
X, Y = np.meshgrid(x, y)

ridge = np.exp(-((X - 0.55) ** 2 / 0.02 + (Y + 0.05) ** 2 / 0.35))
plateau = 0.85 * np.exp(-((X + 0.20) ** 4 / 0.55 + (Y - 0.05) ** 4 / 0.30))
cliff = 1 / (1 + np.exp(8 * (Y - 0.70)))
landscape = (ridge + plateau) * cliff
landscape = landscape / landscape.max()

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(landscape, origin="lower", extent=[x.min(), x.max(), y.min(), y.max()], aspect="auto", vmin=0, vmax=1)
cs = ax.contour(X, Y, landscape, levels=[0.35, 0.55, 0.75], colors="black", linewidths=[1.4, 1.9, 2.4])
ax.clabel(cs, inline=True, fontsize=8)

ax.text(-0.55, 0.05, "safe plateau", fontsize=12)
ax.text(0.54, -0.14, "narrow ridge", fontsize=12)
ax.text(0.32, 0.84, "unstable cliff", fontsize=12)
ax.annotate("good engineering\nwidens this plateau", xy=(-0.35, 0.18), xytext=(-1.05, 0.70),
            arrowprops=dict(arrowstyle="->", linewidth=1.8), fontsize=11)

ax.set_title("Robustness Landscape: Prefer a Plateau Over a Ridge", fontsize=16)
ax.set_xlabel("control parameter 1")
ax.set_ylabel("control parameter 2")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("relative computation support")

savefig(fig, "43_05_robustness_landscape.png")
plt.show()


## 6. Failure tree

The immediate failure is not always logical failure.

Usually the first failure is loss of the connected admissible region.

Logical failure follows after that.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.8))
ax.axis("off")

leaves = [
    ("loss", (0.10, 0.82)),
    ("phase drift", (0.10, 0.64)),
    ("detector noise", (0.10, 0.46)),
    ("timing jitter", (0.10, 0.28)),
    ("feed-forward delay", (0.10, 0.10)),
]

mid = ("loss of connected\nadmissible region", (0.55, 0.46))
out = ("logical failure", (0.88, 0.46))

for label, pos in leaves:
    ax.add_patch(Rectangle((pos[0]-0.085, pos[1]-0.045), 0.17, 0.09, fill=False, linewidth=1.8))
    ax.text(*pos, label, ha="center", va="center", fontsize=9)
    ax.annotate("", xy=(mid[1][0]-0.16, mid[1][1]), xytext=(pos[0]+0.09, pos[1]),
                arrowprops=dict(arrowstyle="->", linewidth=1.3, alpha=0.7))

ax.add_patch(Rectangle((mid[1][0]-0.16, mid[1][1]-0.075), 0.32, 0.15, fill=False, linewidth=2.2))
ax.text(*mid[1], mid[0], ha="center", va="center", fontsize=10)

ax.add_patch(Rectangle((out[1][0]-0.09, out[1][1]-0.045), 0.18, 0.09, fill=False, linewidth=1.8))
ax.text(*out[1], out[0], ha="center", va="center", fontsize=10)
ax.annotate("", xy=(out[1][0]-0.10, out[1][1]), xytext=(mid[1][0]+0.17, mid[1][1]),
            arrowprops=dict(arrowstyle="->", linewidth=2.0))

ax.set_title("Failure Tree: Logical Failure Follows Loss of the Connected Region", fontsize=15)
ax.text(0.5, -0.03, "robustness protects the intermediate engineering object before computation fails",
        ha="center", transform=ax.transAxes, fontsize=10)

savefig(fig, "43_06_failure_tree.png")
plt.show()


## 7. Engineering allocation

The robustness view changes allocation.

Rather than spend effort uniformly, engineering effort follows the bottlenecks that shrink the connected region.


In [ ]:
categories = ["squeezing", "routing", "detection", "timing", "calibration", "stability"]
effort = np.array([26, 20, 18, 12, 12, 12], dtype=float)
effort = effort / effort.sum()

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.pie(effort, labels=categories, autopct=lambda p: f"{p:.0f}%", startangle=90)
ax.set_title("Engineering Priority: Effort Widens Robustness Margins", fontsize=15)

allocation = pd.DataFrame({"category": categories, "allocation_share": effort})
allocation.to_csv(CSV / "43_engineering_allocation.csv", index=False)
allocation.to_json(JS / "43_engineering_allocation.json", orient="records", indent=2)

savefig(fig, "43_07_engineering_allocation.png")
plt.show()
allocation


## 8. Engineering summary

Robustness is the margin surrounding admissible specifications.

It is not a single device metric.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Stage": "Physics",
            "Question": "What can exist?",
            "Engineering object": "Optical resources",
            "Robustness role": "Generate candidate resources",
        },
        {
            "Stage": "Admissibility",
            "Question": "What survives?",
            "Engineering object": "Admitted resources",
            "Robustness role": "Filter failed resources",
        },
        {
            "Stage": "Region",
            "Question": "Where can computation occur?",
            "Engineering object": "Connected admissible region",
            "Robustness role": "Preserve connectivity",
        },
        {
            "Stage": "Robustness",
            "Question": "How much perturbation is tolerated?",
            "Engineering object": "Stability margin",
            "Robustness role": "Maintain operating margin",
        },
        {
            "Stage": "Computation",
            "Question": "Which logical operations remain?",
            "Engineering object": "Fault-tolerant execution",
            "Robustness role": "Support logical computation",
        },
    ]
)

fig, ax = plt.subplots(figsize=(14.5, 4.2))
ax.axis("off")
table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.8)
table.scale(1.08, 1.65)
for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.15)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Robustness Preserves Admissible Computation", fontsize=16, pad=20)
ax.text(0.5, -0.10, "Robust engineering preserves the connected admissible region, not merely the operating point.",
        ha="center", transform=ax.transAxes, fontsize=10.5)

summary.to_csv(CSV / "43_summary.csv", index=False)
summary.to_json(JS / "43_summary.json", orient="records", indent=2)

savefig(fig, "43_08_summary_table.png")
plt.show()
summary


## 9. Export and download

This cell packages all figures and data files into a single zip.


In [ ]:
zip_path = RES / "43_outputs.zip"

files_to_zip = [
    FIG / "43_01_robustness_margin.png",
    FIG / "43_02_drift_directions.png",
    FIG / "43_03_region_survival.png",
    FIG / "43_04_architecture_comparison.png",
    FIG / "43_05_robustness_landscape.png",
    FIG / "43_06_failure_tree.png",
    FIG / "43_07_engineering_allocation.png",
    FIG / "43_08_summary_table.png",
    CSV / "43_region_survival.csv",
    CSV / "43_architecture_comparison.csv",
    CSV / "43_engineering_allocation.csv",
    CSV / "43_summary.csv",
    JS / "43_region_survival.json",
    JS / "43_architecture_comparison.json",
    JS / "43_engineering_allocation.json",
    JS / "43_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- Robustness is a margin around a connected admissible region.
- A high-performing point can be fragile if the surrounding region is narrow.
- Useful architectures preserve region size, connectivity, and margin.
- Engineering effort should follow the bottlenecks that shrink the connected region.
- Fault-tolerant computation depends on preserving admissibility under perturbation.

## Next notebook

A natural Notebook 47 would move from qualitative margins to explicit metrics:

```text
47 — Quantifying Robustness Margins
```

Possible metrics:

- admitted area
- largest connected component size
- distance to failure boundary
- perturbation survival probability
- architecture robustness score
